#Speech Emotion Recognition with MLP Classifier



#Dataset
The Ryerson Audio-Visual Database of Emotional Speech and Song (RAVDESS) 

---
Audio-only files

Audio-only files of all actors (01-24) are available as two separate zip files (~200 MB each):

Speech file (Audio_Speech_Actors_01-24.zip, 215 MB) contains 1440 files: 60 trials per actor x 24 actors = 1440. 
Song file (Audio_Song_Actors_01-24.zip, 198 MB) contains 1012 files: 44 trials per actor x 23 actors = 1012.

Total=2452

---

---
Toronto emotional speech set (TESS)

---


There are a set of 200 target words were spoken in the carrier phrase "Say the word _' by two actresses (aged 26 and 64 years) and recordings were made of the set portraying each of seven emotions (anger, disgust, fear, happiness, pleasant surprise, sadness, and neutral). There are 2800 data points (audio files) in total.

The dataset is organised such that each of the two female actor and their emotions are contain within its own folder. And within that, all 200 target words audio file can be found. The format of the audio file is a WAV format


---



# Mount google drive



# Install following libraries

# Make the necessary imports

In [32]:
import os
import librosa
import numpy as np
import joblib

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix


In [33]:
def extract_features(file_path):
    audio, sr = librosa.load(file_path, duration=3, offset=0.5)
    mfcc = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=40)
    return np.mean(mfcc.T, axis=0)


In [34]:
emotion_map = {
    "01": "neutral",
    "02": "calm",
    "03": "happy",
    "04": "sad",
    "05": "angry",
    "06": "fearful",
    "07": "disgust",
    "08": "surprised"
}

X = []
y = []

dataset_path = "dataset2"  # folder containing Actor_01, Actor_02, ...

for actor_folder in os.listdir(dataset_path):
    actor_path = os.path.join(dataset_path, actor_folder)

    if not os.path.isdir(actor_path):
        continue

    for file in os.listdir(actor_path):
        if not file.endswith(".wav"):
            continue

        file_path = os.path.join(actor_path, file)

        try:
            # Extract emotion code from filename
            emotion_code = file.split("-")[2]
            emotion = emotion_map[emotion_code]

            features = extract_features(file_path)

            X.append(features)
            y.append(emotion)

        except Exception as e:
            print("Error processing:", file_path)


In [5]:
X = np.array(X)
y = np.array(y)

print("Unique labels:", np.unique(y))


Unique labels: ['angry' 'calm' 'disgust' 'fearful' 'happy' 'neutral' 'sad' 'surprised']


In [6]:
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

print(label_encoder.classes_)


['angry' 'calm' 'disgust' 'fearful' 'happy' 'neutral' 'sad' 'surprised']


In [7]:
joblib.dump(label_encoder, "label_encoder.pkl")


['label_encoder.pkl']

In [30]:
model.fit(X_train_smote, y_train_smote)
joblib.dump(model, "mlp_emotion_model.pkl")


NameError: name 'X_train_smote' is not defined

In [11]:
import librosa
import soundfile
import glob
import os, glob, pickle
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score

Define a function extract_feature to extract the mfcc, chroma, and mel features from a sound file. This function takes 4 parameters- the file name and three Boolean parameters for the three features:

* mfcc: Mel Frequency Cepstral Coefficient, represents the short-term power spectrum of a sound
* chroma: Pertains to the 12 different pitch classes
* mel: Mel Spectrogram Frequency

Now, let’s define a dictionary to hold numbers and the emotions available in the RAVDESS & TESS dataset, and a list to hold all 8 emotions- neutral,calm,happy,sad,angry,fearful,disgust,surprised.

In [12]:
import numpy as np
import librosa
import os

def extract_feature(file_name, mfcc, chroma, mel):
    X, sample_rate = librosa.load(file_name, sr=None)  # <-- IMPORTANT CHANGE

    result = np.array([])

    if chroma:
        stft = np.abs(librosa.stft(X))
        chroma_feat = np.mean(librosa.feature.chroma_stft(S=stft, sr=sample_rate).T, axis=0)
        result = np.hstack((result, chroma_feat))

    if mfcc:
        mfcc_feat = np.mean(librosa.feature.mfcc(y=X, sr=sample_rate, n_mfcc=40).T, axis=0)
        result = np.hstack((result, mfcc_feat))

    if mel:
        mel_feat = np.mean(librosa.feature.melspectrogram(y=X, sr=sample_rate).T, axis=0)
        result = np.hstack((result, mel_feat))

    return result


In [13]:
# Emotions in the RAVDESS & TESS dataset
emotions={
  '01':'neutral',
  '02':'calm',
  '03':'happy',
  '04':'sad',
  '05':'angry',
  '06':'fearful',
  '07':'disgust',
  '08':'surprised'
}
# Emotions to observe
observed_emotions=['neutral','calm','happy','sad','angry','fearful', 'disgust','surprised']

# Load the data and extract features for each sound file

In [31]:
def load_data():
    X, y = [], []

    for file in glob.glob('./dataset2/Actor_*/*.wav'):
        file_name = os.path.basename(file)
        emotion = emotions[file_name.split("-")[2]]

        if emotion not in observed_emotions:
            continue

        features = extract_feature(file, mfcc=True, chroma=True, mel=True)
        X.append(features)
        y.append(emotion)

    return np.array(X), np.array(y)


# Split the Dataset
Time to split the dataset into training and testing sets! Let’s keep the test set 25% of everything and use the load_data function for this.

In [15]:
# Split the dataset
import time
x_train, x_test, y_train, y_test = load_data(test_size=0.25)


#Observe the shape of the training and testing datasets:

In [16]:
#Get the shape of the training and testing datasets
print((x_train.shape[0], x_test.shape[0]))

(1080, 360)


# Number of features extracted.

In [17]:
# Get the number of features extracted
print(f'Features extracted: {x_train.shape[1]}')

Features extracted: 180


# MLP Classifier

In [18]:
# Initialize the Multi Layer Perceptron Classifier
model=MLPClassifier(alpha=0.01, batch_size=256, epsilon=1e-08, hidden_layer_sizes=(300,), learning_rate='adaptive', max_iter=500)

#Fit/train the model.

In [19]:
# Train the model
model.fit(x_train,y_train)

MLPClassifier(alpha=0.01, batch_size=256, hidden_layer_sizes=(300,),
              learning_rate='adaptive', max_iter=500)

# Predict the accuracy of our model

Let’s predict the values for the test set. This gives us y_pred (the predicted emotions for the features in the test set).

In [20]:
# Predict for the test set
y_pred=model.predict(x_test)

To calculate the accuracy of our model, we’ll call up the accuracy_score() function we imported from sklearn. Finally, we’ll round the accuracy to 2 decimal places and print it out.

In [21]:
# Calculate the accuracy of our model
accuracy=accuracy_score(y_true=y_test, y_pred=y_pred)
# Print the accuracy
print("Accuracy: {:.2f}%".format(accuracy*100))

Accuracy: 48.06%


#classification Report

In [22]:
from sklearn.metrics import classification_report
print(classification_report(y_test,y_pred))


              precision    recall  f1-score   support

       angry       0.71      0.63      0.67        51
        calm       0.67      0.27      0.38        45
     disgust       0.41      0.74      0.53        43
     fearful       0.68      0.34      0.45        44
       happy       0.55      0.47      0.51        59
     neutral       0.50      0.39      0.44        31
         sad       0.30      0.69      0.42        45
   surprised       0.55      0.26      0.35        42

    accuracy                           0.48       360
   macro avg       0.55      0.47      0.47       360
weighted avg       0.55      0.48      0.48       360



# Confusion Matrix

In [23]:
from sklearn.metrics import confusion_matrix
matrix = confusion_matrix(y_test,y_pred)
print (matrix)

[[32  0  6  2  3  0  4  4]
 [ 0 12 10  0  0  3 20  0]
 [ 2  0 32  0  6  1  2  0]
 [ 2  0  7 15  8  0 12  0]
 [ 4  1  7  5 28  3  8  3]
 [ 1  1  2  0  0 12 15  0]
 [ 2  4  2  0  0  4 31  2]
 [ 2  0 12  0  6  1 10 11]]


#Thank You

In [25]:
from sklearn.neural_network import MLPClassifier

model = MLPClassifier(
    hidden_layer_sizes=(256, 128),
    activation="relu",
    solver="adam",
    max_iter=600,
    early_stopping=True,
    random_state=42
)

model.fit(X_train_smote, y_train_smote)


NameError: name 'X_train_smote' is not defined

In [26]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled,
    y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

print(X_train.shape, y_train.shape)


NameError: name 'X_scaled' is not defined

In [46]:
y_pred = model.predict(X_test)

print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))



              precision    recall  f1-score   support

           0       0.07      0.03      0.04        38
           1       0.00      0.00      0.00        38
           2       0.12      0.03      0.04        38
           3       0.22      0.05      0.08        39
           4       0.18      0.05      0.08        39
           5       0.11      0.05      0.07        19
           6       0.00      0.00      0.00        38
           7       0.00      0.00      0.00        39
           8       0.00      0.00      0.00         0
           9       0.00      0.00      0.00         0
          10       0.00      0.00      0.00         0
          11       0.00      0.00      0.00         0
          12       0.00      0.00      0.00         0
          13       0.00      0.00      0.00         0
          14       0.00      0.00      0.00         0
          15       0.00      0.00      0.00         0
          16       0.00      0.00      0.00         0
          17       0.00    

c:\Users\lenovo\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\lenovo\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\lenovo\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.cap

In [47]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print("Before SMOTE:", np.bincount(y_train))
print("After SMOTE:", np.bincount(y_train_smote))


Before SMOTE: [154 154 154 153 153  77 154 153]
After SMOTE: [154 154 154 154 154 154 154 154]


In [51]:
model = MLPClassifier(
    hidden_layer_sizes=(256, 128),
    activation='relu',
    solver='adam',
    max_iter=600,
    early_stopping=True,
    random_state=42
)

model.fit(X_train_smote, y_train_smote)  # ✅ CORRECT


MLPClassifier(early_stopping=True, hidden_layer_sizes=(256, 128), max_iter=600,
              random_state=42)

In [52]:
from sklearn.metrics import classification_report, confusion_matrix

y_pred = model.predict(X_test)

print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))


Classification Report:
              precision    recall  f1-score   support

       angry       0.76      0.74      0.75        38
        calm       0.73      0.63      0.68        38
     disgust       0.53      0.63      0.58        38
     fearful       0.56      0.72      0.63        39
       happy       0.57      0.41      0.48        39
     neutral       0.35      0.42      0.38        19
         sad       0.55      0.55      0.55        38
   surprised       0.59      0.51      0.55        39

    accuracy                           0.59       288
   macro avg       0.58      0.58      0.57       288
weighted avg       0.59      0.59      0.59       288

Confusion Matrix:
[[28  0  4  2  0  0  2  2]
 [ 0 24  1  0  1  4  8  0]
 [ 3  0 24  3  2  2  1  3]
 [ 0  1  2 28  4  0  1  3]
 [ 2  0  7  8 16  1  1  4]
 [ 0  4  2  0  0  8  4  1]
 [ 0  3  3  3  2  5 21  1]
 [ 4  1  2  6  3  3  0 20]]


In [53]:
def predict_emotion(audio_path):
    features = extract_features(audio_path)
    features = scaler.transform([features])

    prediction = model.predict(features)
    emotion = label_encoder.inverse_transform(prediction)

    return emotion[0]


In [1]:
test_audio = "sad.wav"   # path to your audio file
print("Predicted emotion:", predict_emotion(test_audio))


NameError: name 'predict_emotion' is not defined